<a href="https://colab.research.google.com/github/Aisyaaswy/Pemrograman_Mesin/blob/main/jobsheet_6/dataset_pbl/HNSW/praktikum_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install annoy faiss-cpu hnswlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 10.9 MB/s  0:00:00
  Installing build dependencies ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/19.5 MB ? eta -:--:--  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 30.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 30.8 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64

In [2]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [3]:
# -------------------------------
# Load dataset Sleep Health
# -------------------------------
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')
features = [
    'Age', 'Sleep Duration', 'Quality of Sleep',
    'Physical Activity Level', 'Stress Level', 'Heart Rate', 'Daily Steps'
]
X = df[features].values

# Standarisasi fitur & ubah ke float32 untuk FAISS & HNSW
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

k = 10   # jumlah nearest neighbors

In [4]:
# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.5f} s")

Exact NN done in 0.05999 s


In [5]:
# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.5f} s")

Annoy done in 0.01375 s


In [6]:
# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.5f} s")

HNSW done in 0.03349 s


In [7]:
# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
# nlist disesuaikan menjadi 10 karena total baris Sleep Health berjumlah 374 data
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 10, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 5
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.5f} s")

FAISS IVF done in 0.00882 s


In [8]:
# -------------------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------------------
print("\nTop-5 neighbors for first Sleep Health record:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")


Top-5 neighbors for first Sleep Health record:
Exact NN: [  0 106  69  68  30]
Annoy:    [0, 106, 68, 69, 30]
HNSW:     [  0 106  68  69  30]
FAISS:    [  0 106  68  69  30]


## Analisis Tahap Implementasi (Dataset Sleep Health and Lifestyle)

Tahap ini menguji performa metode pencarian tetangga terdekat pada data kesehatan tidur nyata (374 baris).

- Program memproses dataset dengan mengambil 7 fitur numerik utama (*Age*, *Sleep Duration*, *Quality of Sleep*, *Physical Activity Level*, *Stress Level*, *Heart Rate*, dan *Daily Steps*).
- Sistem menerapkan `StandardScaler` untuk menyamakan skala rentang nilai antar fitur serta mengonversinya menjadi `float32` agar kompatibel dengan pustaka FAISS dan HNSW.
- Parameter klaster `nlist` pada FAISS IVF disesuaikan menjadi 10 agar proporsional dengan jumlah data sebesar 374 baris.
- Hasil eksekusi menunjukkan bahwa ketiga algoritma aproksimasi (Annoy, HNSW, dan FAISS) menghasilkan daftar indeks profil kesehatan tidur terdekat yang konsisten dengan metode acuan Exact NN.